# RET-2 + Dual Residual Adapters (v3) — 1 epoch, no CLIP backprop

**Goal:** Attempt a light-weight embedding improvement with pretrained GeoRSCLIP RET-2. The adapters operate at **retrieval inference**, unlike the reconstruction-only changes in v1/v2. This is a **different module from PR-CLIP reconstruction**, not a claimed PR-CLIP architecture reproduction.

**Protocol:** 1) verify RET-2 baseline near 38.88 on RSICD; 2) split RET-2 training data into 90/10 by unique image ID; 3) cache frozen embeddings; 4) train 3 tiny identity-initialized adapters for **one epoch each**; 5) choose based only on RET-2 held-out split; 6) evaluate on RSICD test once. In the supplied repository `rsicd_val.csv` and `rsicd_test.csv` are identical, so DO NOT use the former for selection.

**No performance guarantee.** The original RET-2 baseline is always preserved and reported. All outputs are saved in a separate Drive directory.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# upload PR-CLIP.zip lên Drive trước, sửa đường dẫn cho đúng
!rm -rf /content/PR-CLIP
!unzip -q -o /content/drive/MyDrive/PR-CLIP.zip -d /content
%cd /content/PR-CLIP

Mounted at /content/drive
/content/PR-CLIP


In [2]:
!pip install -q open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown
!pip install -q "protobuf>=6.31.1,<7"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.85.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
googleapis-common-protos 1.76.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
diffusers 0.42.1 requires huggi

In [3]:
!gdown --fuzzy "https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view" -O images.zip
!rm -rf images && mkdir images
!tar -xzf images.zip -C images
!find images -type f -exec mv -t images {} + 2>/dev/null
!find images -mindepth 1 -type d -empty -delete
!rm images.zip
!mkdir -p cache/weights

Downloading...
From (original): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT
From (redirected): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT&confirm=t&uuid=489d334a-fce5-4f1e-b236-f20afddcd654
To: /content/PR-CLIP/images.zip
100% 1.46G/1.46G [00:19<00:00, 73.9MB/s]


In [4]:
!mkdir -p /content/PR-CLIP/cache/weights
!wget -O /content/PR-CLIP/cache/weights/RET-2.pt \
  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
!ls -la /content/PR-CLIP/cache/weights/RET-2.pt   # khoảng 605MB

--2026-11-03 15:11:06--  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
Resolving huggingface.co (huggingface.co)... 3.171.171.128, 3.171.171.6, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.128|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/655fb2aadc7511e22748819d/48ec26e685f2b7622075158c0649039d64cdcbdbe150faedb6e79ea1d13d0e78?xip=CgYExZO6F8I&X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RS5M_ViT-B-32_RET-2.pt%3B+filename%3D%22RS5M_ViT-B-32_RET-2.pt%22%3B&Expires=1793722266&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU1ZmIyYWFkYzc1MTFlMjI3NDg4MTlkLzQ4ZWMyNmU2ODVmMmI3NjIyMDc1MTU4YzA2NDkwMzlkNjRjZGNiZGJlMTUwZmFlZGI2ZTc5ZWExZDEzZDBlNzhcXD94aXA9Q2dZRXhaTzZGOEkmWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIiwiQ2

In [5]:
%cd /content/PR-CLIP
import os, hashlib
for fn in ['dataset/train/ret2.csv','dataset/test/rsicd_test.csv','dataset/test/rsicd_val.csv']:
    assert os.path.isfile(fn), f'Missing {fn}'
    print(fn, os.path.getsize(fn))
assert os.path.getsize('cache/weights/RET-2.pt') > 500_000_000, 'RET-2 checkpoint looks incomplete'
print('GPU:')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


/content/PR-CLIP
dataset/train/ret2.csv 4862976
dataset/test/rsicd_test.csv 440141
dataset/test/rsicd_val.csv 440141
GPU:
/bin/bash: line 1: nvidia-smi: command not found


## Feature-space adaptation code

This uses the exact RET-2 weights, OpenCLIP ViT-B/32 inference preprocessing, and six Recall@K metrics. Do not change evaluation images/split between comparisons. The dataset and embeddings can be cached for follow-up trials.

In [6]:
from pathlib import Path
p=Path('/content/PR-CLIP/run_adapter_v3.py')
p.write_text('"""PR-CLIP V3: Identity-initialized dual residual adapters on GeoRSCLIP RET-2 features.\nThis is an embedding/adapter improvement (NOT a reconstruction-module modification).\nValidation split: grouped by image within RET-2 training CSV. RSICD val == test in supplied repo.\n"""\nimport os, json, math, random, hashlib\nfrom pathlib import Path\nfrom collections import defaultdict\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom PIL import Image\nfrom tqdm.auto import tqdm\nimport open_clip\n\nROOT=Path(\'/content/PR-CLIP\')\nWEIGHTS=ROOT/\'cache/weights/RET-2.pt\'\nIMAGES=ROOT/\'images\'\nOUT=Path(\'/content/drive/MyDrive/prclip_logs/RET2_DualResidualAdapter_v3\')\nSEED=1\nBATCH_ENCODE=64\nBATCH_TRAIN_IMAGES=128\nDEVICE=\'cuda\' if torch.cuda.is_available() else \'cpu\'\nOUT.mkdir(parents=True,exist_ok=True)\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)\n\nclass ImageDataset(Dataset):\n    def __init__(self,names,preprocess):self.names=list(names);self.preprocess=preprocess\n    def __len__(self):return len(self.names)\n    def __getitem__(self,i):\n        name=self.names[i]\n        p=IMAGES/name\n        if not p.exists():raise FileNotFoundError(f\'Missing image: {p}\')\n        with Image.open(p) as im: return self.preprocess(im.convert(\'RGB\'))\n\ndef uniq_and_map(df):\n    names=list(dict.fromkeys(df[\'filename\'].astype(str)))\n    index={n:i for i,n in enumerate(names)}\n    pairs=torch.tensor([index[x] for x in df[\'filename\'].astype(str)],dtype=torch.long)\n    return names,pairs\n\n@torch.inference_mode()\ndef compute_image_features(names, preprocess, clip):\n    loader=DataLoader(ImageDataset(names,preprocess),batch_size=BATCH_ENCODE,shuffle=False,\n                      num_workers=2,pin_memory=True)\n    rows=[]\n    for x in tqdm(loader,desc=\'Image embeddings\'):\n        feat=clip.encode_image(x.to(DEVICE,non_blocking=True))\n        rows.append(F.normalize(feat.float(),dim=-1).cpu())\n    return torch.cat(rows)\n\n@torch.inference_mode()\ndef compute_text_features(captions,clip):\n    rows=[]\n    for st in tqdm(range(0,len(captions),BATCH_ENCODE),desc=\'Text embeddings\'):\n        t=open_clip.tokenize(captions[st:st+BATCH_ENCODE],context_length=77,truncate=True).to(DEVICE)\n        feat=clip.encode_text(t)\n        rows.append(F.normalize(feat.float(),dim=-1).cpu())\n    return torch.cat(rows)\n\nclass DualResidual(nn.Module):\n    def __init__(self,d=512,bottleneck=128,scale=0.2):\n        super().__init__();self.scale=scale\n        def make():\n            m=nn.Sequential(nn.Linear(d,bottleneck),nn.GELU(),nn.Linear(bottleneck,d))\n            nn.init.zeros_(m[-1].weight);nn.init.zeros_(m[-1].bias)\n            return m\n        self.image_adapter=make();self.text_adapter=make()\n    def adapt_image(self,x): return F.normalize(x+self.scale*self.image_adapter(x),dim=-1)\n    def adapt_text(self,x): return F.normalize(x+self.scale*self.text_adapter(x),dim=-1)\n\ndef score_mr(image_emb,text_emb,text_image_ids,bs=512):\n    """Exact binary-hit Recall@K, same semantics as repository\'s retrieval evaluation."""\n    image_emb=F.normalize(image_emb.float(),dim=-1).to(DEVICE)\n    text_emb=F.normalize(text_emb.float(),dim=-1).to(DEVICE)\n    ids=text_image_ids.to(DEVICE)\n    ks=(1,5,10)\n    hit_i={k:0 for k in ks};hit_t={k:0 for k in ks}\n    for st in range(0,len(image_emb),bs):\n        sim=image_emb[st:st+bs] @ text_emb.T\n        top=sim.topk(10,dim=1).indices\n        good=ids[top].eq(torch.arange(st,st+len(top),device=DEVICE)[:,None])\n        for k in ks:hit_i[k]+=good[:,:k].any(dim=1).sum().item()\n    for st in range(0,len(text_emb),bs):\n        sim=text_emb[st:st+bs] @ image_emb.T\n        top=sim.topk(10,dim=1).indices\n        good=top.eq(ids[st:st+len(top),None])\n        for k in ks:hit_t[k]+=good[:,:k].any(dim=1).sum().item()\n    vals={f\'retrieval-image2text-R@{k}\':round(100*hit_i[k]/len(image_emb),2) for k in ks}\n    vals.update({f\'retrieval-text2image-R@{k}\':round(100*hit_t[k]/len(text_emb),2) for k in ks})\n    vals[\'retrieval-mean-recall\']=round(sum(vals.values())/6,2)\n    return vals\n\n@torch.inference_mode()\ndef adapted_features(net,ii,tt):\n    net.eval()\n    # batch to keep memory consumption predictable\n    i=torch.cat([net.adapt_image(chunk.to(DEVICE)).cpu() for chunk in ii.split(4096)])\n    t=torch.cat([net.adapt_text(chunk.to(DEVICE)).cpu() for chunk in tt.split(4096)])\n    return i,t\n\ndef train_one_epoch(net,train_image,train_text,train_map,lr=3e-4,seed=SEED):\n    """Multi-positive CLIP loss, 1 unique image per batch; each image has all its captions."""\n    torch.manual_seed(seed);rng=np.random.default_rng(seed)\n    groups=defaultdict(list)\n    for j,i in enumerate(train_map.tolist()):groups[int(i)].append(j)\n    ids=np.array(sorted(groups))\n    rng.shuffle(ids)\n    net.train()\n    optim=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=0.01)\n    running=[]\n    for st in range(0,len(ids),BATCH_TRAIN_IMAGES):\n        selected=ids[st:st+BATCH_TRAIN_IMAGES]\n        if len(selected)<2:continue\n        capids=[j for k in selected for j in groups[int(k)]]\n        capids_t=torch.as_tensor(capids,dtype=torch.long)\n        image=train_image[torch.as_tensor(selected,dtype=torch.long)].to(DEVICE)\n        text=train_text[capids_t].to(DEVICE)\n        image_vec=net.adapt_image(image)\n        text_vec=net.adapt_text(text)\n        # Clip temperature is fixed conservatively; regularize changes from strong RET-2 embeddings.\n        logits=(image_vec@text_vec.T)/0.07\n        row_index={int(k):j for j,k in enumerate(selected)}\n        tgt=torch.tensor([row_index[int(train_map[k])] for k in capids],device=DEVICE)\n        txt_loss=F.cross_entropy(logits.T,tgt)\n        positive=F.one_hot(tgt,num_classes=len(selected)).T.float()\n        positive=positive/positive.sum(1,keepdim=True).clamp_min(1)\n        img_loss=-(positive*F.log_softmax(logits,dim=1)).sum(1).mean()\n        drift=(image_vec-image).square().sum(1).mean()+(text_vec-text).square().sum(1).mean()\n        loss=0.5*(txt_loss+img_loss)+0.1*drift\n        optim.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1.0);optim.step()\n        running.append(float(loss.item()))\n    return float(np.mean(running))\n\ndef cache_or_build(key,names,df,preprocess,clip):\n    path=OUT/f\'{key}_features.pt\'\n    digest=hashlib.sha256((\'\\n\'.join(names)+\'\\0\'+\'\\n\'.join(df.title.astype(str))).encode()).hexdigest()\n    if path.exists():\n        blob=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if blob.get(\'digest\')==digest:\n            print(\'Cache hit:\',path)\n            return blob[\'image\'].float(),blob[\'text\'].float()\n    images=compute_image_features(names,preprocess,clip)\n    texts=compute_text_features(df.title.astype(str).tolist(),clip)\n    torch.save({\'digest\':digest,\'image\':images.half(),\'text\':texts.half()},path)\n    return images,texts\n\ndef main():\n    assert DEVICE==\'cuda\', \'GPU runtime required\'\n    train_csv=ROOT/\'dataset/train/ret2.csv\'\n    rsicd_csv=ROOT/\'dataset/test/rsicd_test.csv\'\n    rsicd_val=ROOT/\'dataset/test/rsicd_val.csv\'\n    for p in (WEIGHTS,train_csv,rsicd_csv):assert p.exists(),f\'Missing {p}\'\n    if rsicd_val.exists() and rsicd_val.read_bytes()==rsicd_csv.read_bytes():\n        print(\'WARNING: RSICD val and test are identical in your source. Never select on this val!\')\n    print(\'Loading GeoRSCLIP RET-2...\')\n    clip,_,preprocess=open_clip.create_model_and_transforms(\'ViT-B-32\',pretrained=None,device=DEVICE)\n    checkpoint=torch.load(WEIGHTS,map_location=\'cpu\',weights_only=False)\n    if isinstance(checkpoint,dict) and \'state_dict\' in checkpoint:checkpoint=checkpoint[\'state_dict\']\n    result=clip.load_state_dict(checkpoint,strict=True)\n    print(\'Checkpoint loaded:\',result)\n    clip.eval()\n    for p in clip.parameters():p.requires_grad_(False)\n    df=pd.read_csv(train_csv).dropna(subset=[\'filename\',\'title\']).reset_index(drop=True)\n    eval_df=pd.read_csv(rsicd_csv,encoding=\'gb18030\').dropna(subset=[\'filename\',\'title\']).reset_index(drop=True)\n    all_images,all_map=uniq_and_map(df)\n    test_images,test_map=uniq_and_map(eval_df)\n    print(\'Train\',len(all_images),\'unique images,\',len(df),\'captions; RSICD\',len(test_images),\'images,\',len(eval_df),\'captions\')\n    all_i,all_t=cache_or_build(\'RET2_train\',all_images,df,preprocess,clip)\n    test_i,test_t=cache_or_build(\'RSICD_test\',test_images,eval_df,preprocess,clip)\n    base_test=score_mr(test_i,test_t,test_map)\n    print(\'RET-2 baseline test:\',base_test)\n    # Numerical parity guard protects against comparing a broken evaluation pipeline.\n    assert abs(base_test[\'retrieval-mean-recall\']-38.88)<0.7, (\n        \'Baseline does not match your measured 38.88. Check weights, transforms and dataset before proceeding.\')\n    # Group split is by image; each caption of the same image must remain in the same split.\n    perm=np.random.default_rng(SEED).permutation(len(all_images))\n    n_val=max(1,round(len(all_images)*0.1))\n    vi_ids=perm[:n_val];tr_ids=perm[n_val:]\n    def subset(image_ids):\n        old_to_new={int(old):j for j,old in enumerate(image_ids)}\n        cap_mask=torch.tensor([int(x) in old_to_new for x in all_map.tolist()],dtype=torch.bool)\n        cap_inds=cap_mask.nonzero().flatten()\n        local_map=torch.tensor([old_to_new[int(k)] for k in all_map[cap_inds].tolist()],dtype=torch.long)\n        return all_i[torch.as_tensor(image_ids)],all_t[cap_inds],local_map\n    tri,trt,trm=subset(tr_ids);vai,vat,vam=subset(vi_ids)\n    base_val=score_mr(vai,vat,vam)\n    print(\'Independent RET2 held-out baseline:\',base_val)\n    results=[]\n    for lr in (1e-4,3e-4,1e-3):\n        net=DualResidual(d=all_i.shape[1]).to(DEVICE)\n        initial_i,initial_t=adapted_features(net,vai,vat)\n        assert torch.allclose(initial_i,vai,atol=1e-5) and torch.allclose(initial_t,vat,atol=1e-5)\n        loss=train_one_epoch(net,tri,trt,trm,lr=lr)\n        iv,tv=adapted_features(net,vai,vat)\n        val=score_mr(iv,tv,vam)\n        ck=OUT/f\'adapter_lr{lr:.0e}.pt\'\n        torch.save({\'model\':net.cpu().state_dict(),\'lr\':lr,\'heldout_val\':val,\'heldout_baseline\':base_val,\n                    \'train_loss\':loss,\'checkpoint\':str(WEIGHTS),\'seed\':SEED,\'bottleneck\':128,\'scale\':0.2},ck)\n        net.to(DEVICE)\n        results.append((lr,val[\'retrieval-mean-recall\'],str(ck)))\n        print(f\'LR={lr:.0e}, train loss={loss:.4f}, heldout mR={val["retrieval-mean-recall"]:.2f}\')\n    # Safety gate: an adapter is not selected if it fails to improve *held-out* RET2 validation.\n    best=max(results,key=lambda x:x[1]);improved=best[1]>base_val[\'retrieval-mean-recall\']\n    if improved:\n        saved=torch.load(best[2],map_location=\'cpu\',weights_only=False)\n        net=DualResidual(d=all_i.shape[1],bottleneck=saved[\'bottleneck\'],scale=saved[\'scale\']).to(DEVICE)\n        net.load_state_dict(saved[\'model\'])\n        ti,tt=adapted_features(net,test_i,test_t)\n        final=score_mr(ti,tt,test_map)\n        selected=f\'adapter_lr{best[0]:.0e}\'\n    else:\n        final=base_test;selected=\'identity_RET2_no_adapter\'\n    report={\'method\':\'frozen GeoRSCLIP feature extraction + identity residual adapter\',\n            \'heldout_validation_split\':\'10% of RET-2 images, image-group separated\',\n            \'original_RET2_test\':base_test,\'heldout_baseline\':base_val,\n            \'candidate_heldout_mr\':[{\'lr\':lr,\'mr\':mr} for lr,mr,_ in results],\n            \'selected_model\':selected,\'selected_test\':final,\n            \'caveat\':\'Adapter selected on RET-2 holdout, not RSICD. No guaranteed RSICD gain.\'}\n    (OUT/\'v3_results.json\').write_text(json.dumps(report,indent=2,ensure_ascii=False))\n    print(json.dumps(report,indent=2,ensure_ascii=False))\n    print(\'Saved results to\',OUT)\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
print('Script saved:',p,'bytes=',p.stat().st_size)
import ast
ast.parse(p.read_text())
print('Python syntax: OK')

Script saved: /content/PR-CLIP/run_adapter_v3.py bytes= 11779
Python syntax: OK


## Run once

The first execution computes and caches embeddings; reruns reuse caches. This notebook does **not** perform 20 epochs of end-to-end CLIP fine-tuning. It only trains a lightweight residual adapter for one pass through unique training images per candidate learning rate.

In [7]:
%cd /content/PR-CLIP
!python run_adapter_v3.py


/content/PR-CLIP
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Traceback (most recent call last):
  File "/content/PR-CLIP/run_adapter_v3.py", line 226, in <module>
    if __name__=='__main__':main()
                            ~~~~^^
  File "/content/PR-CLIP/run_adapter_v3.py", line 151, in main
    assert DEVICE=='cuda', 'GPU runtime required'
           ^^^^^^^^^^^^^^
AssertionError: GPU runtime required


In [8]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/prclip_logs/RET2_DualResidualAdapter_v3/v3_results.json')
if p.exists():
    result=json.loads(p.read_text())
    print('Original RET-2 mR:',result['original_RET2_test']['retrieval-mean-recall'])
    print('Held-out baseline mR:',result['heldout_baseline']['retrieval-mean-recall'])
    print('Candidate validation:',result['candidate_heldout_mr'])
    print('Selected:',result['selected_model'])
    print('RSICD test mR:',result['selected_test']['retrieval-mean-recall'])
else: print('Run training cell first.')


Run training cell first.


### Scientific interpretation

- If `selected_model = identity_RET2_no_adapter`, none of the adapter candidates improved on the held-out RET-2 split; do **not** claim an improvement.
- If an adapter was selected but RSICD test falls below 38.88, the adaptation did not generalize from held-out RET-2 to RSICD.
- The paper's PR-CLIP mR is 39.23, but this adapter is a **separate method**, not the original cross-modal positional reconstruction module.
- For a rigorous report, repeat promising settings across seeds **without** choosing them using the test set. This notebook intentionally uses a cheap first-pass screening protocol.